# Jalon A - Probe Workflow\n\nNotebook guide pour un workflow complet avec `extract_dataset_residuals` et probes (lineaire + MLP).

In [ ]:
import torch\nfrom torch.utils.data import DataLoader, TensorDataset\n\nfrom boardGPT.nn import GPT, GPTConfig\nfrom boardGPT.interpretability import (\n    extract_dataset_residuals,\n    LinearBoardProbe,\n    MLPBoardProbe,\n    fit_probe,\n    eval_probe,\n)

In [ ]:
# 1) Tiny GPT pour demo\ncfg = GPTConfig(block_size=10, vocab_size=24, n_layer=2, n_head=2, n_embd=20, dropout=0.0, bias=True)\nmodel = GPT(cfg, use_flash=False)\n\ntokens = torch.randint(0, cfg.vocab_size, (300, cfg.block_size))\ntoken_loader = DataLoader(TensorDataset(tokens, torch.zeros(300, 1)), batch_size=50, shuffle=False)

In [ ]:
# 2) Extraction d'activations sur la couche 1, position last\nacts = extract_dataset_residuals(\n    model=model,\n    batches=token_loader,\n    layer=1,\n    position_policy='last',\n    device='cpu',\n)\nprint('activations:', acts.shape)

In [ ]:
# 3) Labels synthetiques pour entrainer un probe\n#    (dans un vrai run, remplacer par des labels issus de `move_ids_to_board_labels`)\nW = torch.randn(acts.shape[1], 64, 3)\ny = torch.einsum('nd,dck->nck', acts, W).argmax(dim=-1)\n\ntrain_x, test_x = acts[:240], acts[240:]\ntrain_y, test_y = y[:240], y[240:]\ntrain_loader = DataLoader(TensorDataset(train_x, train_y), batch_size=32, shuffle=True)\ntest_loader = DataLoader(TensorDataset(test_x, test_y), batch_size=32, shuffle=False)

In [ ]:
# 4) Probe lineaire\nlin_probe = LinearBoardProbe(input_dim=acts.shape[1])\n_ = fit_probe(lin_probe, train_loader, val_loader=test_loader, epochs=10, learning_rate=3e-3, device='cpu')\nlin_stats = eval_probe(lin_probe, test_loader, device='cpu')\nprint('linear acc:', lin_stats['acc_global'])

In [ ]:
# 5) Probe MLP\nmlp_probe = MLPBoardProbe(input_dim=acts.shape[1], hidden_dim=64)\n_ = fit_probe(mlp_probe, train_loader, val_loader=test_loader, epochs=10, learning_rate=3e-3, device='cpu')\nmlp_stats = eval_probe(mlp_probe, test_loader, device='cpu')\nprint('mlp acc:', mlp_stats['acc_global'])